# Vectors and matrices
**Quantum Computing · Week 3 (The maths of qubits, and interference) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qc/w3d1.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

States are column vectors, gates are matrices, and running a circuit is matrix multiplication.
With NumPy you can simulate a qubit yourself, and check it against Qiskit.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
psi = 0.6 * ket0 + 0.8j * ket1           # a superposition is just a weighted sum of columns
print("|psi> =", psi)
print("normalized?", np.isclose(np.vdot(psi, psi).real, 1.0))

## A gate is a matrix; applying it is multiplication
`@` is NumPy's matrix product. The columns of a gate are where $|0\rangle$ and $|1\rangle$ go.

In [ ]:
s = 1 / np.sqrt(2)
X = np.array([[0, 1], [1, 0]], dtype=complex)
H = np.array([[s, s], [s, -s]], dtype=complex)
S = np.array([[1, 0], [0, 1j]], dtype=complex)

print("H|0> =", np.round(H @ ket0, 3), "  = first column of H:", np.round(H[:, 0], 3))
print("H|1> =", np.round(H @ ket1, 3), " = second column of H:", np.round(H[:, 1], 3))
print("H|psi> =", np.round(H @ psi, 3))
print("same as 0.6·H|0> + 0.8i·H|1>?", np.allclose(H @ psi, 0.6 * (H @ ket0) + 0.8j * (H @ ket1)))

## Our matrices are Qiskit's matrices

In [ ]:
for name, M in [("x", X), ("h", H), ("s", S)]:
    qc = QuantumCircuit(1)
    getattr(qc, name)(0)
    print(f"{name.upper()}: same as Qiskit's? {np.allclose(Operator(qc).data, M)}")

## A circuit is a product of matrices, read right to left
The circuit "H, then S" is the matrix $S\,H$: the gate applied first sits next to the state.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)
print(qc.draw("text"))
print("S @ H matches the circuit?", np.allclose(Operator(qc).data, S @ H))
print("H @ S matches the circuit?", np.allclose(Operator(qc).data, H @ S))
print("state from the circuit:", np.round(Statevector(qc).data, 3), "  from S @ H @ |0>:", np.round(S @ H @ ket0, 3))

## Your turn: one matrix for a whole circuit
The circuit applies S, then H, then T (with $T = \mathrm{diag}(1, e^{i\pi/4})$). Build its matrix with NumPy in the
right order, check it against Qiskit's `Operator`, and check that the wrong order gives something else.

In [ ]:
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)
qc = QuantumCircuit(1)
qc.s(0)
qc.h(0)
qc.t(0)
U = np.eye(2)            # TODO: the product of T, H and S in the right order
print("matches Qiskit?", np.allclose(Operator(qc).data, U))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
U = T @ H @ S            # S acts first, so it sits on the right
wrong = S @ H @ T
print("T @ H @ S matches Qiskit?", np.allclose(Operator(qc).data, U))
print("S @ H @ T matches Qiskit?", np.allclose(Operator(qc).data, wrong))
print("U =\n", np.round(U, 3))